# 02 Procesamiento de texto, recuperación secuencial e indexación

## Objetivo

Implementar y comparar dos mecanismos de recuperación de información —búsqueda secuencial e índice invertido— sobre colecciones de libros del Proyecto Gutenberg, investigando experimentalmente su rendimiento y escalabilidad.

### Parte A. Corpus pequeño

**1. Preparación del corpus**

Utilice los 18 documentos de la colección Gutenberg disponible en NLTK.

1. Cargue los textos originales.
2. Identifique el número de documentos y sus tamaños.
3. Implemente tokenización y normalización: conversión a minúsculas, tratamiento de puntuación y caracteres especiales.

Seleccione y documente una configuración de preprocesamiento que se utilizará consistentemente en todos los experimentos.

In [1]:
import re
import time
import nltk
from nltk.corpus import gutenberg

nltk.download('gutenberg', quiet=True)

# 1. Carga de los textos originales
textos = {doc_id: gutenberg.raw(doc_id) for doc_id in gutenberg.fileids()}

# 3. Tokenización y normalización
def preprocesar(texto):
    texto = texto.lower()                         # todo a minúsculas
    texto = re.sub(r"[^a-z0-9]+", " ", texto)     # puntuación y caracteres especiales -> espacio
    return texto.split()

docs = {doc_id: preprocesar(t) for doc_id, t in textos.items()}

# 2. Número de documentos y tamaños
print("Número de documentos:", len(docs))
print(f"{'Documento':25s} {'Caracteres':>12s} {'Tokens':>10s}")
for doc_id in docs:
    print(f"{doc_id:25s} {len(textos[doc_id]):>12,} {len(docs[doc_id]):>10,}")
print("Total de tokens:", sum(len(t) for t in docs.values()))

Número de documentos: 18
Documento                   Caracteres     Tokens
austen-emma.txt                887,071    161,980
austen-persuasion.txt          466,292     84,167
austen-sense.txt               673,022    120,787
bible-kjv.txt                4,332,554    854,046
blake-poems.txt                 38,153      6,936
bryant-stories.txt             249,439     46,699
burgess-busterbrown.txt         84,663     16,359
carroll-alice.txt              144,395     27,336
chesterton-ball.txt            457,450     82,869
chesterton-brown.txt           406,629     73,288
chesterton-thursday.txt        320,525     58,729
edgeworth-parents.txt          935,158    170,799
melville-moby_dick.txt       1,242,990    218,619
milton-paradise.txt            468,220     80,497
shakespeare-caesar.txt         112,310     20,875
shakespeare-hamlet.txt         162,881     30,271
shakespeare-macbeth.txt        100,351     18,351
whitman-leaves.txt             711,215    126,605
Total de tokens: 2199213


**Configuración de preprocesamiento elegida** (se usa igual en todo el notebook):

- Se pasa todo el texto a minúsculas, así `Whale` y `whale` cuentan como el mismo término.
- Todo lo que no sea letra o dígito (`a-z`, `0-9`) se cambia por un espacio y luego se separa por espacios. Así se quitan la puntuación y los caracteres especiales.
- No se eliminan stopwords ni se aplica stemming, para que las dos búsquedas trabajen exactamente con los mismos tokens.

Una consecuencia es que las contracciones se separan (`don't` → `don`, `t`), pero para esta práctica no es un problema.

**2. Búsqueda secuencial**

Implemente una función que reciba un término de consulta y devuelva los identificadores de todos los documentos que lo contienen con una búsqueda secuencial.

La implementación deberá utilizar obligatoriamente dos lazos `for` anidados: el primero recorre los documentos y el segundo los tokens de cada documento.

No es necesario utilizar índices invertidos, motores de búsqueda ni estructuras auxiliares que almacenen anticipadamente los documentos asociados a cada término.

Ejecute consultas con términos frecuentes, poco frecuentes e inexistentes.

In [2]:
def busqueda_secuencial(termino, docs):
    termino = termino.lower()
    resultado = []
    for doc_id, tokens in docs.items():      # lazo 1: documentos
        for token in tokens:                 # lazo 2: tokens del documento
            if token == termino:
                resultado.append(doc_id)
                break                        # ya está en este doc, pasamos al siguiente
    return resultado

consultas = {
    "frecuente":      ["the", "love", "sea"],
    "poco frecuente": ["macbeth", "alice", "whale"],
    "inexistente":    ["internet", "algorithm", "smartphone"],
}

for tipo, terminos in consultas.items():
    print(f"--- {tipo} ---")
    for t in terminos:
        res = busqueda_secuencial(t, docs)
        print(f"{t:12s} -> {len(res):2d} docs {res}")

--- frecuente ---
the          -> 18 docs ['austen-emma.txt', 'austen-persuasion.txt', 'austen-sense.txt', 'bible-kjv.txt', 'blake-poems.txt', 'bryant-stories.txt', 'burgess-busterbrown.txt', 'carroll-alice.txt', 'chesterton-ball.txt', 'chesterton-brown.txt', 'chesterton-thursday.txt', 'edgeworth-parents.txt', 'melville-moby_dick.txt', 'milton-paradise.txt', 'shakespeare-caesar.txt', 'shakespeare-hamlet.txt', 'shakespeare-macbeth.txt', 'whitman-leaves.txt']
love         -> 15 docs ['austen-emma.txt', 'austen-persuasion.txt', 'austen-sense.txt', 'bible-kjv.txt', 'blake-poems.txt', 'bryant-stories.txt', 'burgess-busterbrown.txt', 'carroll-alice.txt', 'chesterton-ball.txt', 'chesterton-brown.txt', 'chesterton-thursday.txt', 'edgeworth-parents.txt', 'melville-moby_dick.txt', 'milton-paradise.txt', 'whitman-leaves.txt']
sea          -> 16 docs ['austen-emma.txt', 'austen-persuasion.txt', 'bible-kjv.txt', 'blake-poems.txt', 'bryant-stories.txt', 'carroll-alice.txt', 'chesterton-ball.txt', 'c

Los términos frecuentes (`the`, `love`, `sea`) aparecen en casi todos los documentos, los poco frecuentes solo en unos pocos (`macbeth` solo en la obra de Shakespeare) y los inexistentes devuelven una lista vacía. El `break` hace que, cuando el término aparece pronto, no se revise el resto del documento.

**3. Medición de tiempos de ejecución**

Utilice `time.perf_counter_ns()` o una herramienta de medición equivalente para obtener el tiempo de ejecución de las consultas.

Cada medición deberá corresponder a **30 ejecuciones de la misma consulta**. Y se debe calcular la media aritmética.

Registre los resultados en milisegundos. No incluya la carga de archivos ni la tokenización dentro del tiempo de consulta.

In [3]:
def medir_tiempo(funcion, termino, estructura, repeticiones=30):
    tiempos = []
    for _ in range(repeticiones):
        inicio = time.perf_counter_ns()
        funcion(termino, estructura)
        fin = time.perf_counter_ns()
        tiempos.append((fin - inicio) / 1e6)   # ns -> ms
    return sum(tiempos) / len(tiempos)

print(f"{'Término':12s} {'Tipo':16s} {'Secuencial (ms)':>16s}")
for tipo, terminos in consultas.items():
    for t in terminos:
        ms = medir_tiempo(busqueda_secuencial, t, docs)
        print(f"{t:12s} {tipo:16s} {ms:>16.4f}")

Término      Tipo              Secuencial (ms)
the          frecuente                  0.0087
love         frecuente                  2.4335
sea          frecuente                  4.7305
macbeth      poco frecuente            36.7015
alice        poco frecuente            33.1821
whale        poco frecuente            20.1251
internet     inexistente               32.4266
algorithm    inexistente               32.4618
smartphone   inexistente               32.1309


Solo se mide la llamada a la función de búsqueda; la carga y la tokenización ya se hicieron antes. Los términos inexistentes son el peor caso, porque obligan a recorrer **todos** los tokens de la colección. Con `the` pasa lo contrario: aparece al principio de cada documento y el `break` corta casi de inmediato.

**4. Construcción del índice invertido**

Implemente un índice invertido que contenga el vocabulario de la colección y las listas de documentos asociados a cada término.

Inspeccione las listas de postings y calcule la frecuencia documental (DF) de varios términos.

Implemente consultas de un término y consultas booleanas con AND, OR y NOT.

In [4]:
def construir_indice(docs):
    indice = {}
    for doc_id, tokens in docs.items():
        for token in set(tokens):            # cada término una sola vez por documento
            indice.setdefault(token, []).append(doc_id)
    return indice

indice = construir_indice(docs)
todos_docs = list(docs.keys())
print("Tamaño del vocabulario:", len(indice))

# Postings y frecuencia documental (DF)
for t in ["the", "love", "whale", "alice", "macbeth", "internet"]:
    postings = indice.get(t, [])
    print(f"{t:10s} DF={len(postings):2d} {postings}")

# Consultas de un término y booleanas
def buscar_indice(termino, indice):
    return indice.get(termino.lower(), [])

def AND(a, b):
    return sorted(set(a) & set(b))

def OR(a, b):
    return sorted(set(a) | set(b))

def NOT(a):
    return sorted(set(todos_docs) - set(a))

w = lambda t: buscar_indice(t, indice)
print("\nwhale              :", w("whale"))
print("whale AND sea      :", AND(w("whale"), w("sea")))
print("alice OR macbeth   :", OR(w("alice"), w("macbeth")))
print("NOT love           :", NOT(w("love")))
print("king AND NOT queen :", AND(w("king"), NOT(w("queen"))))

Tamaño del vocabulario: 41840
the        DF=18 ['austen-emma.txt', 'austen-persuasion.txt', 'austen-sense.txt', 'bible-kjv.txt', 'blake-poems.txt', 'bryant-stories.txt', 'burgess-busterbrown.txt', 'carroll-alice.txt', 'chesterton-ball.txt', 'chesterton-brown.txt', 'chesterton-thursday.txt', 'edgeworth-parents.txt', 'melville-moby_dick.txt', 'milton-paradise.txt', 'shakespeare-caesar.txt', 'shakespeare-hamlet.txt', 'shakespeare-macbeth.txt', 'whitman-leaves.txt']
love       DF=15 ['austen-emma.txt', 'austen-persuasion.txt', 'austen-sense.txt', 'bible-kjv.txt', 'blake-poems.txt', 'bryant-stories.txt', 'burgess-busterbrown.txt', 'carroll-alice.txt', 'chesterton-ball.txt', 'chesterton-brown.txt', 'chesterton-thursday.txt', 'edgeworth-parents.txt', 'melville-moby_dick.txt', 'milton-paradise.txt', 'whitman-leaves.txt']
whale      DF= 6 ['bible-kjv.txt', 'bryant-stories.txt', 'chesterton-ball.txt', 'melville-moby_dick.txt', 'shakespeare-hamlet.txt', 'whitman-leaves.txt']
alice      DF= 3 ['ca

El índice es un diccionario `término → lista de documentos` (postings). Como los documentos se recorren en orden, cada lista queda ordenada. La DF es simplemente la longitud de la lista: `the` tiene DF = 18 (está en todos), mientras que `macbeth` tiene DF = 1. Las operaciones booleanas se resuelven con intersección (AND), unión (OR) y complemento respecto al total de documentos (NOT).

**5. Comparación**

Ejecute las mismas consultas con ambos métodos.

Compruebe que recuperan exactamente los mismos identificadores de documentos y mida el tiempo medio de 30 ejecuciones para cada consulta y método.

In [5]:
print(f"{'Término':12s} {'Iguales':>8s} {'Secuencial (ms)':>16s} {'Índice (ms)':>12s} {'Veces más rápido':>17s}")
for tipo, terminos in consultas.items():
    for t in terminos:
        r_seq = busqueda_secuencial(t, docs)
        r_idx = buscar_indice(t, indice)
        iguales = sorted(r_seq) == sorted(r_idx)

        t_seq = medir_tiempo(busqueda_secuencial, t, docs)
        t_idx = medir_tiempo(buscar_indice, t, indice)
        print(f"{t:12s} {str(iguales):>8s} {t_seq:>16.4f} {t_idx:>12.6f} {t_seq / t_idx:>17.0f}")

Término       Iguales  Secuencial (ms)  Índice (ms)  Veces más rápido
the              True           0.0053     0.000167                32
love             True           2.2078     0.000250              8831
sea              True           4.2576     0.000257             16588
macbeth          True          33.9769     0.000340             99932
alice            True          33.0561     0.000367             90153
whale            True          21.6996     0.000290             74826
internet         True          33.0713     0.000283            116722
algorithm        True          33.8410     0.000277            122317
smartphone       True          32.1552     0.000293            109620


**Conclusión de la Parte A:** los dos métodos devuelven exactamente los mismos documentos para todas las consultas, así que el índice no cambia el resultado, solo la velocidad. La búsqueda secuencial depende de cuántos tokens tenga que revisar (en el peor caso todos, O(T)), mientras que con el índice la consulta es un acceso al diccionario, prácticamente constante. La diferencia es de varios órdenes de magnitud, sobre todo en los términos raros o inexistentes. El costo del índice está en construirlo una vez al principio y en la memoria que ocupa.

### Parte B. Corpus grande

**6. Descarga de Gutenberg**

Descargue el archivo oficial `txt-files.tar.zip`, de aproximadamente 11 GB, desde:

https://www.gutenberg.org/cache/epub/feeds/

Identifique los archivos TXT, evite contar distintas versiones del mismo libro como documentos independientes y registre la cantidad de documentos válidos.

**7. Recuperación sobre el corpus grande**

Reutilice las funciones desarrolladas en la Parte A.

1. Procese los documentos con la misma configuración de normalización.
2. Ejecute las consultas mediante búsqueda secuencial y mida los tiempos.
3. Construya el índice invertido.
4. Repita las consultas utilizando el índice.
5. Compruebe la igualdad de resultados y compare los tiempos.

Trabaje con subconjuntos de la colección de tamaños crecientes, conservando los identificadores originales. Documente los recursos de memoria y almacenamiento utilizados.

### Parte C. Análisis experimental de escalabilidad

**8. Tiempo de ejecución en función del corpus**

Seleccione al menos cinco tamaños crecientes del corpus grande. Como referencia, pueden utilizarse 50, 100, 250, 500 y 1.000 documentos, ampliándolos si el equipo lo permite.

Para cada tamaño $n$:

1. Ejecute una misma consulta mediante búsqueda secuencial.
2. Repítala 30 veces y calcule el tiempo promedio.
3. Construya el índice invertido correspondiente a ese subconjunto.
4. Ejecute la consulta mediante el índice, también 30 veces.
5. Registre ambos tiempos y el número total de tokens.

Elabore una **gráfica del tiempo medio de ejecución en función del número de documentos**, colocando el número de documentos en el eje X y el tiempo en milisegundos en el eje Y. Represente los dos métodos en la misma gráfica.

**9. Estimación de una función de tiempo**

Utilizando los resultados experimentales, determine una función:

$t=f(n)$

para cada método de recuperación.

Como primera aproximación, utilice un ajuste lineal por mínimos cuadrados:

$t(n)=an+b$

Estime los parámetros $a$ y $b$, reporte el coeficiente de determinación $R^2$ y represente las funciones ajustadas junto a los datos experimentales.

Explique si el modelo lineal representa adecuadamente las mediciones y cómo se relaciona con la complejidad computacional de cada algoritmo.